# 02 Feature Engineering

This notebook creates leakage-safe customer features. The feature window contains only historical transactions, while the prediction window is used only to create the churn label.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import CLEANED_TRANSACTIONS_PATH, CUSTOMER_FEATURES_PATH
from src.feature_engineering import build_customer_features, validate_customer_features

## Load Clean Transactions

`cleaned_transactions.csv` should come from the preprocessing step. It contains valid customer purchases and the `Revenue` field.

In [ ]:
transactions = pd.read_csv(CLEANED_TRANSACTIONS_PATH, parse_dates=['InvoiceDate'])
transactions.head()

## Build Features and Churn Label

RFM converts purchase history into customer-level behavior. Churn is defined as no purchase during the future 90-day prediction window.

In [ ]:
features, metadata = build_customer_features(transactions)
features.to_csv(CUSTOMER_FEATURES_PATH, index=False)
metadata

In [ ]:
validate_customer_features(features)

## Feature Meaning

- `Recency`: days since latest purchase before cutoff.
- `Frequency`: number of orders before cutoff.
- `Monetary`: total customer revenue before cutoff.
- `AverageOrderValue`: revenue per order.
- `TotalItems`: purchased quantity.
- `UniqueProducts`: product diversity.
- `CustomerLifetimeDays`: customer age inside the feature window.
- `AverageDaysBetweenOrders`: average purchase gap.
- `Churn`: future inactivity label, not an input feature.